# Actividad en clase en casa 4
###### O sea que no es en clase.

Este notebook implementa un esquema RSA para firma digital y verificación entre dos participantes anónimos con identidades secretas:

- Mr. _¿Sí? ¿Clarito?_
- Dr. F3R M1N

Cada uno genera su propio par de llaves y firma su propio mensaje usando su llave privada. Después, el otro verifica el mensaje usando la llave pública correspondiente.

El notebook también guarda evidencias en archivos `.txt` separados dentro de la carpeta `evidencias`.

## Decisiones técnicas

### Decisión 1: esquema implementado y razón de elección
Implementamos un esquema RSA, para firma digital y verificacion esto fue para explicar la relacion publica y privada de las llaves y en la prueba en vivo explicar mas facilmente las operaciones matematicas asi tengamos nervios o algo y no confundirnos facilmente, la geneacion de primos, el calculo de n y phi, el exponenete publico y privado, asi mismo la razon fue por es asimetrico esto se adapta al flujo de la tarea

La parte del código que corresponde a esta decisión se encuentra principalmente en la función `generar_llaves`. Ahí se generan los valores `p` y `q`, se calcula `n`, se calcula `phi`, se define `e` y se obtiene `d` usando el inverso modular. También se relaciona con las funciones `firmar_mensaje` y `verificar_firma`, porque ahí se usa la llave privada para firmar y la llave pública para verificar.

### Decisión 2: forma de representación interna del mensaje
Primero hacerlo en texto y luego convertirlo a bytes con UFT-8, para luego con SHA-256 hacerlo en numero entero y meterlo al mododulo de n.

RSA no lee texto por eso le preparamos las cosas para sus necesidades.

La parte del código que corresponde a esta decisión está en la función `preparar_mensaje`. En esa función el mensaje se codifica con `mensaje.encode("utf-8")`, después se calcula el hash usando `hashlib.sha256`, luego se convierte el hash a entero con `int(hash_hexadecimal, 16)` y finalmente se obtiene `valor_preparado = hash_entero % n`.

### Decisión 3: uso del mensaje completo o de una transformación antes de firmar
Al final decidimos no firmar el mensaje completo, esto fue por que nos acordamos de la clase donde el metodo era buscar una palabra o algo conocido para empezar a decifrar el mensaje pero en este caso pensamos que estaria mejor que fuera en el mensjae convertido en hasj y que ya este ajustado por el modulo de n.
Si el mensaje cambia por poco o mucho el hash no seria valido, esto nos permite ver si el mensaje coincide con el valor o no.

La parte del código que corresponde a esta decisión también está en `preparar_mensaje`, porque ahí se genera el hash. Después se usa en `firmar_mensaje`, donde se firma `preparado["valor_preparado"]` con la llave privada. También se comprueba en `verificar_firma`, porque el programa vuelve a preparar el mensaje recibido y compara su valor preparado contra el valor recuperado desde la firma.

Esta decisión se demuestra en las pruebas de mensajes alterados. En esas pruebas se usa la firma original, pero el mensaje se cambia. Como el mensaje alterado produce otro hash, la verificación falla.

### Decisión 4: simplificaciones realizadas
Primero: los primos se hacen dentro del notebook, no se usan librerias hechas para esto.
Segundo se usa random que eso es muy malo para un caso real.
Tercero se guardan cosas como las llaves, mensajes, firmas y resultados como txt.
Cuarto no se hacen un padding seguro.
Las partes del código que corresponden a esta decisión son `es_probablemente_primo`, `generar_primo`, `generar_llaves`, `guardar_txt` y las secciones donde se crean los archivos de evidencia. También se relaciona con las carpetas `persona_1`, `persona_2`, `intercambio` y `pruebas_invalidas`, porque esas evidencias permiten mostrar el funcionamiento del sistema sin depender solo de los resultados impresos en pantalla.

### Decisión 5: debilidades o limitaciones de la implementación
La implementacion tiene debilidade y bastante claras la mayoria se reducen a simplificaciones realizadas, en si serian que no tienen todas las protecciones y pasos necesarios para resolverlo en un entorno real es solo para la tarea y para defenderla.

La parte del código que corresponde a estas limitaciones está en la generación de primos y llaves, especialmente en `generar_primo` y `generar_llaves`. También se observa en las funciones que guardan evidencias, porque se guardan valores privados solo para demostrar cómo funciona el proceso. La conclusión técnica del notebook también menciona que el sistema sirve para aprender y defender la actividad, pero no para proteger información real.

En resumen, estas decisiones permiten que el notebook muestre un criptosistema asimétrico funcional, entendible y defendible. El código genera llaves, prepara mensajes, firma, verifica, guarda evidencias y demuestra casos inválidos como mensajes alterados, llaves incorrectas y firmas mal formadas.

## 1. Importación de módulos permitidos

Vamos a usar solo los siguientes módulos, profe.

Como puede ver, no hay nada de generación automática de llaves, de firmas ni verificación automática. `hashlib` se usa únicamente para calcular SHA-256 del mensaje.
###### Es como cuando un mago muestra que no trae nada en las manos pero al final sí trae algo en la manga.

In [1]:
from pathlib import Path
import hashlib
import math
import random

## 2. Carpeta de evidencias

Esta sección crea una carpeta llamada `evidencias`.

Dentro de ella se separan los archivos de Mr. _¿Sí? ¿Clarito?_, Dr. F3R M1N, intercambio y pruebas inválidas. Estos archivos sirven para demostrar el funcionamiento del sistema paso por paso.

In [2]:
CARPETA_EVIDENCIAS = Path("evidencias")

CARPETA_MR_SI_CLARITO = CARPETA_EVIDENCIAS / "mr_si_clarito"
CARPETA_DR_F3R_M1N = CARPETA_EVIDENCIAS / "dr_f3r_m1n"
CARPETA_INTERCAMBIO = CARPETA_EVIDENCIAS / "intercambio"
CARPETA_PRUEBAS_INVALIDAS = CARPETA_EVIDENCIAS / "pruebas_invalidas"

for carpeta in [
    CARPETA_MR_SI_CLARITO,
    CARPETA_DR_F3R_M1N,
    CARPETA_INTERCAMBIO,
    CARPETA_PRUEBAS_INVALIDAS
]:
    carpeta.mkdir(parents=True, exist_ok=True)


def guardar_txt(ruta, contenido):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(str(contenido), encoding="utf-8")
    print(f"Archivo generado: {ruta}")

## 3. Funciones matemáticas auxiliares
El algoritmo extendido de Euclides permite calcular el inverso modular. Ese inverso modular se usa para obtener el exponente privado `d`, que forma parte de la llave privada.

In [3]:
def algoritmo_extendido_euclides(a, b):
    if b == 0:
        return a, 1, 0

    mcd, x1, y1 = algoritmo_extendido_euclides(b, a % b)
    x = y1
    y = x1 - (a // b) * y1

    return mcd, x, y


def inverso_modular(a, modulo):
    mcd, x, _ = algoritmo_extendido_euclides(a, modulo)

    if mcd != 1:
        raise ValueError("No existe inverso modular porque los valores no son coprimos.")

    return x % modulo

## 4. Generación de primos
###### Waos, o sea que así es como luego salen primos hasta de debajo de las piedras cuando uno va a los bautizos.

Para generar llaves RSA se necesitan dos números primos secretos.

Usaremos la prueba probabilística de Miller-Rabin que investigamos en una de las tareas.
###### Atención: Sus alumnos David y Rafael no son expertos en ciberseguridad, no replique el siguiente código en un entorno de producción ya que es posible que dejemos un enorme hueco de vulnerabilidades.

In [4]:
def es_probablemente_primo(numero, rondas=12):
    if numero < 2:
        return False

    primos_pequenos = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31]

    if numero in primos_pequenos:
        return True

    for primo in primos_pequenos:
        if numero % primo == 0:
            return False

    d = numero - 1
    s = 0

    while d % 2 == 0:
        d //= 2
        s += 1

    for _ in range(rondas):
        a = random.randrange(2, numero - 2)
        x = pow(a, d, numero)

        if x == 1 or x == numero - 1:
            continue

        for _ in range(s - 1):
            x = pow(x, 2, numero)

            if x == numero - 1:
                break
        else:
            return False

    return True


def generar_primo(bits=256):
    while True:
        candidato = random.getrandbits(bits)
        candidato |= (1 << bits - 1)
        candidato |= 1

        if es_probablemente_primo(candidato):
            return candidato

## 5. Generación de llaves

Cada persona genera su propio par de llaves.

La llave pública está formada por `(e, n)` y puede compartirse.

La llave privada está formada por `(d, n)` y debe mantenerse en privado si no serían llave publica 2.

También se guardan `p`, `q` y `phi`, pero solo como evidencia educativa. En un caso real esos valores no deberían andar paseándose por ahí en archivos de texto.

In [5]:
def generar_llaves(bits=256):
    p = generar_primo(bits)
    q = generar_primo(bits)

    while p == q:
        q = generar_primo(bits)

    n = p * q
    phi = (p - 1) * (q - 1)

    e = 65537

    if math.gcd(e, phi) != 1:
        e = 3

        while math.gcd(e, phi) != 1:
            e += 2

    d = inverso_modular(e, phi)

    llave_publica = {
        "e": e,
        "n": n
    }

    llave_privada = {
        "d": d,
        "n": n
    }

    detalles = {
        "p": p,
        "q": q,
        "phi": phi
    }

    return llave_publica, llave_privada, detalles

## 6. Preparación del mensaje

RSA no trabaja directamente con texto, así que primero tenemos que convertir el mensaje a algo que el algoritmo pueda manejar.

El mensaje se pasa a bytes usando UTF-8, luego se calcula su hash con SHA-256 y ese hash se convierte a un número entero. Después se reduce con módulo `n`, porque RSA trabaja dentro de ese espacio numérico.

Lo importante aquí es que si el mensaje cambia aunque sea poquito, el hash cambia. Entonces la firma no coincide y la verificación falla.

In [6]:
def preparar_mensaje(mensaje, n):
    if not isinstance(mensaje, str):
        raise TypeError("El mensaje debe ser una cadena de texto.")

    mensaje_bytes = mensaje.encode("utf-8")
    hash_hexadecimal = hashlib.sha256(mensaje_bytes).hexdigest()
    hash_entero = int(hash_hexadecimal, 16)
    valor_preparado = hash_entero % n

    return {
        "mensaje_original": mensaje,
        "hash_hexadecimal": hash_hexadecimal,
        "hash_entero": hash_entero,
        "valor_preparado": valor_preparado
    }

## 7. Firma digital

Aquí cada participante firma su propio mensaje usando su llave privada.

La firma es un número calculado a partir del valor preparado del mensaje y del exponente privado `d`.

Por eso la llave privada no se comparte. Si alguien más la tuviera, podría generar firmas como si fuera esa persona.

###### En pocas palabras: aquí se pone el sello personal, pero versión matemática.

In [7]:
def firmar_mensaje(mensaje, llave_privada):
    if not isinstance(llave_privada, dict):
        raise TypeError("La llave privada debe tener formato de diccionario.")

    if "d" not in llave_privada or "n" not in llave_privada:
        raise ValueError("La llave privada debe contener los valores d y n.")

    d = llave_privada["d"]
    n = llave_privada["n"]

    preparado = preparar_mensaje(mensaje, n)
    firma = pow(preparado["valor_preparado"], d, n)

    return firma, preparado

## 8. Verificación de firma

Para verificar una firma se usa la llave pública de quien supuestamente firmó el mensaje.

El programa toma la firma, aplica la operación con la llave pública y recupera un valor numérico. Ese valor se compara con el valor preparado del mensaje recibido.

Si los dos valores coinciden, la firma es válida. Si no coinciden, significa que algo no cuadra: el mensaje fue cambiado, la firma no corresponde o se usó una llave equivocada.

###### Aquí no se adivina nada, o coincide o nos equivocamos.

In [8]:
def verificar_firma(mensaje, firma, llave_publica):
    try:
        if not isinstance(llave_publica, dict):
            return False, "La llave pública no tiene un formato válido.", None

        if "e" not in llave_publica or "n" not in llave_publica:
            return False, "La llave pública debe contener los valores e y n.", None

        if not isinstance(firma, int):
            return False, "La firma debe ser un número entero.", None

        e = llave_publica["e"]
        n = llave_publica["n"]

        if firma < 0 or firma >= n:
            return False, "La firma está fuera del rango válido.", None

        preparado = preparar_mensaje(mensaje, n)
        valor_recuperado = pow(firma, e, n)

        datos = {
            "hash_mensaje_recibido": preparado["hash_hexadecimal"],
            "valor_preparado_mensaje_recibido": preparado["valor_preparado"],
            "valor_recuperado_desde_firma": valor_recuperado
        }

        if valor_recuperado == preparado["valor_preparado"]:
            return True, "La firma es válida.", datos

        return False, "La firma no corresponde al mensaje o a la llave pública usada.", datos

    except Exception as error:
        return False, f"Error durante la verificación: {error}", None

## 9. Formatos para guardar evidencias

Estas funciones no hacen la parte criptográfica principal, pero ayudan a que el trabajo se pueda revisar mejor.

Lo que hacen es convertir las llaves, mensajes, hashes, firmas y resultados en texto entendible para guardarlos en archivos `.txt`.

Así no queda todo perdido en la salida del notebook y se puede abrir cada evidencia por separado.

###### Porque si todo se imprime junto, solo será cuestión de leer para saber todo cosa bastante improbable en la actualidad.

In [9]:
def formato_llave_publica(nombre, llave):
    return f'''Llave pública de {nombre}

Esta llave puede compartirse.
Sirve para verificar firmas creadas con la llave privada correspondiente.

e = {llave["e"]}

n = {llave["n"]}
'''


def formato_llave_privada(nombre, llave):
    return f'''Llave privada de {nombre}

Esta llave debe mantenerse en secreto.
Sirve para firmar mensajes.

d = {llave["d"]}

n = {llave["n"]}
'''


def formato_detalles(nombre, detalles):
    return f'''Detalles educativos de generación de llaves de {nombre}

Estos valores se muestran solo para explicar cómo se generaron las llaves.
En un sistema real no deberían compartirse.

p = {detalles["p"]}

q = {detalles["q"]}

phi = {detalles["phi"]}
'''


def formato_hash(nombre, preparado):
    return f'''Preparación del mensaje de {nombre}

Mensaje original:
{preparado["mensaje_original"]}

Hash SHA-256:
{preparado["hash_hexadecimal"]}

Hash convertido a entero:
{preparado["hash_entero"]}

Valor preparado dentro del módulo n:
{preparado["valor_preparado"]}
'''


def formato_firma(nombre, firma):
    return f'''Firma digital de {nombre}

La firma fue generada usando la llave privada de {nombre}.
Esta firma se comparte junto con el mensaje y la llave pública para que otra persona pueda verificarla.

firma = {firma}
'''


def formato_verificacion(titulo, mensaje, firma, llave_publica, resultado, explicacion, datos):
    texto = f'''{titulo}

Mensaje recibido:
{mensaje}

Firma recibida:
{firma}

Llave pública usada:
e = {llave_publica.get("e", "no disponible")}
n = {llave_publica.get("n", "no disponible")}

Resultado:
{resultado}

Explicación:
{explicacion}
'''

    if datos:
        texto += f'''
Hash del mensaje recibido:
{datos["hash_mensaje_recibido"]}

Valor preparado del mensaje recibido:
{datos["valor_preparado_mensaje_recibido"]}

Valor recuperado desde la firma usando la llave pública:
{datos["valor_recuperado_desde_firma"]}

Comparación:
La firma es válida solo si el valor recuperado desde la firma coincide con el valor preparado del mensaje recibido.
'''

    return texto

## 10. Generación de llaves para Mr. _¿Sí? ¿Clarito?_ y Dr. F3R M1N

Aquí se generan dos pares de llaves independientes.

Mr. _¿Sí? ¿Clarito?_ tiene su propia llave pública y su propia llave privada. Dr. F3R M1N también tiene las suyas.

Esto es importante porque cada participante debe firmar con su llave privada y el otro solo debe necesitar la llave pública para verificar.


In [10]:
random.seed(132)

llave_publica_1, llave_privada_1, detalles_1 = generar_llaves(bits=256)
llave_publica_2, llave_privada_2, detalles_2 = generar_llaves(bits=256)

guardar_txt(CARPETA_MR_SI_CLARITO / "01_llave_publica_mr_si_clarito.txt", formato_llave_publica("Mr. _¿Sí? ¿Clarito?_", llave_publica_1))
guardar_txt(CARPETA_MR_SI_CLARITO / "02_llave_privada_mr_si_clarito.txt", formato_llave_privada("Mr. _¿Sí? ¿Clarito?_", llave_privada_1))
guardar_txt(CARPETA_MR_SI_CLARITO / "03_detalles_generacion_mr_si_clarito.txt", formato_detalles("Mr. _¿Sí? ¿Clarito?_", detalles_1))

guardar_txt(CARPETA_DR_F3R_M1N / "01_llave_publica_dr_f3r_m1n.txt", formato_llave_publica("Dr. F3R M1N", llave_publica_2))
guardar_txt(CARPETA_DR_F3R_M1N / "02_llave_privada_dr_f3r_m1n.txt", formato_llave_privada("Dr. F3R M1N", llave_privada_2))
guardar_txt(CARPETA_DR_F3R_M1N / "03_detalles_generacion_dr_f3r_m1n.txt", formato_detalles("Dr. F3R M1N", detalles_2))

print("Llaves generadas correctamente para Mr. _¿Sí? ¿Clarito?_ y Dr. F3R M1N.")

Archivo generado: evidencias/mr_si_clarito/01_llave_publica_mr_si_clarito.txt
Archivo generado: evidencias/mr_si_clarito/02_llave_privada_mr_si_clarito.txt
Archivo generado: evidencias/mr_si_clarito/03_detalles_generacion_mr_si_clarito.txt
Archivo generado: evidencias/dr_f3r_m1n/01_llave_publica_dr_f3r_m1n.txt
Archivo generado: evidencias/dr_f3r_m1n/02_llave_privada_dr_f3r_m1n.txt
Archivo generado: evidencias/dr_f3r_m1n/03_detalles_generacion_dr_f3r_m1n.txt
Llaves generadas correctamente para Mr. _¿Sí? ¿Clarito?_ y Dr. F3R M1N.


## 11. Mensajes de Mr. _¿Sí? ¿Clarito?_ y Dr. F3R M1N

En esta sección cada participante escribe un mensaje diferente.

La idea es que no sea una sola persona firmando y verificando todo, sino simular un intercambio entre dos personas.

Cada mensaje se guardará en un archivo para que luego se pueda ver exactamente qué fue lo que se firmó.

In [11]:
mensaje_1 = "Estimado, Doctor. Disculpe la hora, le escribo para recomendarle un posdoctorado de Algoritmos Cuánticos. [enlace]"
mensaje_2 = "Gracias, lo reviso. Me encuentro un poco rebasado en actividades con mis alumnos que no me ponenen atención."

guardar_txt(CARPETA_MR_SI_CLARITO / "04_mensaje_mr_si_clarito.txt", f"Mensaje de Mr. _¿Sí? ¿Clarito?_:\n\n{mensaje_1}\n")
guardar_txt(CARPETA_DR_F3R_M1N / "04_mensaje_dr_f3r_m1n.txt", f"Mensaje de Dr. F3R M1N:\n\n{mensaje_2}\n")

print("Mensaje de Mr. _¿Sí? ¿Clarito?_:")
print(mensaje_1)
print()
print("Mensaje de Dr. F3R M1N:")
print(mensaje_2)

Archivo generado: evidencias/mr_si_clarito/04_mensaje_mr_si_clarito.txt
Archivo generado: evidencias/dr_f3r_m1n/04_mensaje_dr_f3r_m1n.txt
Mensaje de Mr. _¿Sí? ¿Clarito?_:
Estimado, Doctor. Disculpe la hora, le escribo para recomendarle un posdoctorado de Algoritmos Cuánticos. [enlace]

Mensaje de Dr. F3R M1N:
Gracias, lo reviso. Me encuentro un poco rebasado en actividades con mis alumnos que no me ponenen atención.


## 12. Firma de los mensajes

Aquí Mr. _¿Sí? ¿Clarito?_ firma su mensaje con su propia llave privada.

Después Dr. F3R M1N hace lo mismo con su mensaje y su llave privada.

También se guarda el hash de cada mensaje y la firma generada, porque eso ayuda a demostrar que el mensaje fue preparado antes de firmarse.


In [12]:
firma_1, preparado_1 = firmar_mensaje(mensaje_1, llave_privada_1)
firma_2, preparado_2 = firmar_mensaje(mensaje_2, llave_privada_2)

guardar_txt(CARPETA_MR_SI_CLARITO / "05_hash_mensaje_mr_si_clarito.txt", formato_hash("Mr. _¿Sí? ¿Clarito?_", preparado_1))
guardar_txt(CARPETA_MR_SI_CLARITO / "06_firma_mr_si_clarito.txt", formato_firma("Mr. _¿Sí? ¿Clarito?_", firma_1))

guardar_txt(CARPETA_DR_F3R_M1N / "05_hash_mensaje_dr_f3r_m1n.txt", formato_hash("Dr. F3R M1N", preparado_2))
guardar_txt(CARPETA_DR_F3R_M1N / "06_firma_dr_f3r_m1n.txt", formato_firma("Dr. F3R M1N", firma_2))

print("Firma de Mr. _¿Sí? ¿Clarito?_:")
print(firma_1)
print()
print("Firma de Dr. F3R M1N:")
print(firma_2)

Archivo generado: evidencias/mr_si_clarito/05_hash_mensaje_mr_si_clarito.txt
Archivo generado: evidencias/mr_si_clarito/06_firma_mr_si_clarito.txt
Archivo generado: evidencias/dr_f3r_m1n/05_hash_mensaje_dr_f3r_m1n.txt
Archivo generado: evidencias/dr_f3r_m1n/06_firma_dr_f3r_m1n.txt
Firma de Mr. _¿Sí? ¿Clarito?_:
3677733261125552768686566752127254627310254431324306414847450479987055629178645259916071704889377783107482980708344430275074748300608979631877955407622832

Firma de Dr. F3R M1N:
2024522749569809511616625247481643708672188735493051880748112849113348501003803029901466144250193990759514734034804429408682685142860839641479226028852811


## 13. Intercambio y verificación correcta

En esta parte ocurre el caso ideal.

Mr. _¿Sí? ¿Clarito?_ recibe el mensaje y la firma de Dr. F3R M1N, y los verifica usando la llave pública de Dr. F3R M1N.

Luego Dr. F3R M1N hace lo mismo con el mensaje de Mr. _¿Sí? ¿Clarito?_ usando la llave pública de Mr. _¿Sí? ¿Clarito?_.

Si todo está correcto, ambas verificaciones deben salir como válidas.

In [13]:
resultado_p1_verifica_p2, explicacion_p1_verifica_p2, datos_p1_verifica_p2 = verificar_firma(
    mensaje_2,
    firma_2,
    llave_publica_2
)

resultado_p2_verifica_p1, explicacion_p2_verifica_p1, datos_p2_verifica_p1 = verificar_firma(
    mensaje_1,
    firma_1,
    llave_publica_1
)

guardar_txt(
    CARPETA_INTERCAMBIO / "01_mr_si_clarito_verifica_a_dr_f3r_m1n.txt",
    formato_verificacion(
        "Mr. _¿Sí? ¿Clarito?_ verifica el mensaje de Dr. F3R M1N",
        mensaje_2,
        firma_2,
        llave_publica_2,
        resultado_p1_verifica_p2,
        explicacion_p1_verifica_p2,
        datos_p1_verifica_p2
    )
)

guardar_txt(
    CARPETA_INTERCAMBIO / "02_dr_f3r_m1n_verifica_a_mr_si_clarito.txt",
    formato_verificacion(
        "Dr. F3R M1N verifica el mensaje de Mr. _¿Sí? ¿Clarito?_",
        mensaje_1,
        firma_1,
        llave_publica_1,
        resultado_p2_verifica_p1,
        explicacion_p2_verifica_p1,
        datos_p2_verifica_p1
    )
)

print("Mr. _¿Sí? ¿Clarito?_ verifica a Dr. F3R M1N:")
print(resultado_p1_verifica_p2, explicacion_p1_verifica_p2)
print()
print("Dr. F3R M1N verifica a Mr. _¿Sí? ¿Clarito?_:")
print(resultado_p2_verifica_p1, explicacion_p2_verifica_p1)

Archivo generado: evidencias/intercambio/01_mr_si_clarito_verifica_a_dr_f3r_m1n.txt
Archivo generado: evidencias/intercambio/02_dr_f3r_m1n_verifica_a_mr_si_clarito.txt
Mr. _¿Sí? ¿Clarito?_ verifica a Dr. F3R M1N:
True La firma es válida.

Dr. F3R M1N verifica a Mr. _¿Sí? ¿Clarito?_:
True La firma es válida.


## 14. Valores recuperados desde las firmas

Aquí vale la pena aclarar algo importante: no se está recuperando el mensaje original.

Lo que se recupera desde la firma es un valor numérico. Ese valor debe coincidir con el valor preparado del mensaje recibido.

Si coincide, la firma se acepta. Si no coincide, el programa la rechaza.

###### No es “desencriptar el mensaje”, es más como revisar si el sello sí pertenece a esa carta.


In [14]:
guardar_txt(
    CARPETA_INTERCAMBIO / "03_valor_recuperado_firma_mr_si_clarito.txt",
    f'''Valor recuperado desde la firma de Mr. _¿Sí? ¿Clarito?_

Al verificar la firma de Mr. _¿Sí? ¿Clarito?_ con su llave pública, se obtiene este valor:

{datos_p2_verifica_p1["valor_recuperado_desde_firma"]}

Este valor debe compararse contra el valor preparado del mensaje recibido:

{datos_p2_verifica_p1["valor_preparado_mensaje_recibido"]}
'''
)

guardar_txt(
    CARPETA_INTERCAMBIO / "04_valor_recuperado_firma_dr_f3r_m1n.txt",
    f'''Valor recuperado desde la firma de Dr. F3R M1N

Al verificar la firma de Dr. F3R M1N con su llave pública, se obtiene este valor:

{datos_p1_verifica_p2["valor_recuperado_desde_firma"]}

Este valor debe compararse contra el valor preparado del mensaje recibido:

{datos_p1_verifica_p2["valor_preparado_mensaje_recibido"]}
'''
)

print("Valores recuperados guardados correctamente.")

Archivo generado: evidencias/intercambio/03_valor_recuperado_firma_mr_si_clarito.txt
Archivo generado: evidencias/intercambio/04_valor_recuperado_firma_dr_f3r_m1n.txt
Valores recuperados guardados correctamente.


## 15. Resumen del intercambio

Se guarda un resumen general del intercambio correcto.

Este archivo sirve para mostrar que ambas personas pudieron verificar el mensaje de la otra usando solo la llave pública correspondiente, el mensaje y la firma.

In [15]:
guardar_txt(
    CARPETA_INTERCAMBIO / "05_resumen_intercambio.txt",
    f'''Resumen del intercambio

Mr. _¿Sí? ¿Clarito?_ firma su propio mensaje con su llave privada.
Dr. F3R M1N firma su propio mensaje con su llave privada.

Mr. _¿Sí? ¿Clarito?_ verifica el mensaje de Dr. F3R M1N usando la llave pública de Dr. F3R M1N:
{resultado_p1_verifica_p2} - {explicacion_p1_verifica_p2}

Dr. F3R M1N verifica el mensaje de Mr. _¿Sí? ¿Clarito?_ usando la llave pública de Mr. _¿Sí? ¿Clarito?_:
{resultado_p2_verifica_p1} - {explicacion_p2_verifica_p1}

La llave privada no se comparte.
Lo que se comparte para verificar es el mensaje, la firma y la llave pública.
'''
)

print("Resumen del intercambio generado.")

Archivo generado: evidencias/intercambio/05_resumen_intercambio.txt
Resumen del intercambio generado.


## 16. Prueba inválida: mensaje de Mr. _¿Sí? ¿Clarito?_ alterado

Ahora probamos qué pasa si el mensaje de Mr. _¿Sí? ¿Clarito?_ se modifica después de haber sido firmado.

Se usa la firma original, pero con un mensaje cambiado. Como el contenido no sería el mismo, el hash tampoco será el mismo.

El resultado esperado es que la verificación falle.

###### Aquí el sistema actúa como maestro revisando tarea copiada: “esto no es tu nombre, ni matrícula es la de otro compañero”, pero lo subiste tú.


In [16]:
mensaje_1_alterado = "No quedó clarito."

resultado_mensaje_1_alterado, explicacion_mensaje_1_alterado, datos_mensaje_1_alterado = verificar_firma(
    mensaje_1_alterado,
    firma_1,
    llave_publica_1
)

guardar_txt(CARPETA_PRUEBAS_INVALIDAS / "01_mensaje_mr_si_clarito_alterado.txt", f"Mensaje alterado de Mr. _¿Sí? ¿Clarito?_:\n\n{mensaje_1_alterado}\n")

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "02_verificacion_mensaje_mr_si_clarito_alterado.txt",
    formato_verificacion(
        "Verificación del mensaje alterado de Mr. _¿Sí? ¿Clarito?_",
        mensaje_1_alterado,
        firma_1,
        llave_publica_1,
        resultado_mensaje_1_alterado,
        explicacion_mensaje_1_alterado,
        datos_mensaje_1_alterado
    )
)

print(resultado_mensaje_1_alterado, explicacion_mensaje_1_alterado)

Archivo generado: evidencias/pruebas_invalidas/01_mensaje_mr_si_clarito_alterado.txt
Archivo generado: evidencias/pruebas_invalidas/02_verificacion_mensaje_mr_si_clarito_alterado.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 17. Prueba inválida: mensaje de Dr. F3R M1N alterado

Esta prueba hace lo mismo, pero ahora con el mensaje de Dr. F3R M1N.

Se conserva su firma original, pero se cambia el texto del mensaje. Entonces, al verificar, el valor recuperado desde la firma no coincide con el valor preparado del mensaje alterado.

Por eso el resultado debe ser inválido.


In [17]:
mensaje_2_alterado = "Ustedes pueden pensar."

resultado_mensaje_2_alterado, explicacion_mensaje_2_alterado, datos_mensaje_2_alterado = verificar_firma(
    mensaje_2_alterado,
    firma_2,
    llave_publica_2
)

guardar_txt(CARPETA_PRUEBAS_INVALIDAS / "03_mensaje_dr_f3r_m1n_alterado.txt", f"Mensaje alterado de Dr. F3R M1N:\n\n{mensaje_2_alterado}\n")

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "04_verificacion_mensaje_dr_f3r_m1n_alterado.txt",
    formato_verificacion(
        "Verificación del mensaje alterado de Dr. F3R M1N",
        mensaje_2_alterado,
        firma_2,
        llave_publica_2,
        resultado_mensaje_2_alterado,
        explicacion_mensaje_2_alterado,
        datos_mensaje_2_alterado
    )
)

print(resultado_mensaje_2_alterado, explicacion_mensaje_2_alterado)

Archivo generado: evidencias/pruebas_invalidas/03_mensaje_dr_f3r_m1n_alterado.txt
Archivo generado: evidencias/pruebas_invalidas/04_verificacion_mensaje_dr_f3r_m1n_alterado.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 18. Prueba inválida: llave pública incorrecta

Aquí se intenta verificar la firma de Mr. _¿Sí? ¿Clarito?_ usando la llave pública de Dr. F3R M1N.

Esto debe fallar porque la firma fue generada con la llave privada de Mr. _¿Sí? ¿Clarito?_, así que solo debe verificarse correctamente con su llave pública correspondiente.

###### Es como intentar abrir la puerta de tu casa con la llave del Carro: mucha fe, pero no.


In [18]:
resultado_llave_incorrecta, explicacion_llave_incorrecta, datos_llave_incorrecta = verificar_firma(
    mensaje_1,
    firma_1,
    llave_publica_2
)

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "05_verificacion_con_llave_publica_incorrecta.txt",
    formato_verificacion(
        "Verificación con llave pública incorrecta",
        mensaje_1,
        firma_1,
        llave_publica_2,
        resultado_llave_incorrecta,
        explicacion_llave_incorrecta,
        datos_llave_incorrecta
    )
)

print(resultado_llave_incorrecta, explicacion_llave_incorrecta)

Archivo generado: evidencias/pruebas_invalidas/05_verificacion_con_llave_publica_incorrecta.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 19. Prueba inválida: firma mal formada

Aquí se intenta verificar una firma.

El resultado esperado es que el programa no se rompa y responda con un mensaje de error controlado.

In [19]:
firma_mal_formada = "firma_no_numerica"

resultado_firma_mal_formada, explicacion_firma_mal_formada, datos_firma_mal_formada = verificar_firma(
    mensaje_1,
    firma_mal_formada,
    llave_publica_1
)

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "06_firma_mal_formada.txt",
    f'''Prueba con firma mal formada

Mensaje:
{mensaje_1}

Firma enviada:
{firma_mal_formada}

Resultado:
{resultado_firma_mal_formada}

Explicación:
{explicacion_firma_mal_formada}
'''
)

print(resultado_firma_mal_formada, explicacion_firma_mal_formada)

Archivo generado: evidencias/pruebas_invalidas/06_firma_mal_formada.txt
False La firma debe ser un número entero.


## 19. Prueba inválida: firma mal formada

En esta prueba se manda una firma que ni siquiera tiene el formato correcto.

En vez de usar un número, se manda texto. El programa debe detectar eso y responder con un error controlado, no romperse ni explotar como si fuera práctica de laboratorio mal hecha.



In [20]:
firma_no_corresponde = firma_2

resultado_firma_no_corresponde, explicacion_firma_no_corresponde, datos_firma_no_corresponde = verificar_firma(
    mensaje_1,
    firma_no_corresponde,
    llave_publica_1
)

guardar_txt(
    CARPETA_PRUEBAS_INVALIDAS / "07_firma_que_no_corresponde.txt",
    formato_verificacion(
        "Verificación con firma que no corresponde al mensaje",
        mensaje_1,
        firma_no_corresponde,
        llave_publica_1,
        resultado_firma_no_corresponde,
        explicacion_firma_no_corresponde,
        datos_firma_no_corresponde
    )
)

print(resultado_firma_no_corresponde, explicacion_firma_no_corresponde)

Archivo generado: evidencias/pruebas_invalidas/07_firma_que_no_corresponde.txt
False La firma no corresponde al mensaje o a la llave pública usada.


## 21. Resumen final de pruebas

Esta sección junta los resultados principales.

Los casos correctos deben aparecer como `True`, porque ahí el mensaje, la firma y la llave pública sí corresponden.

Los casos inválidos deben aparecer como `False`, porque algo fue alterado, mal formado o verificado con la llave equivocada.

Este resumen sirve para comprobar rápido que el sistema no solo funciona cuando todo está bien, sino que también rechaza errores.


In [21]:
pruebas = [
    ("Mr. _¿Sí? ¿Clarito?_ verifica a Dr. F3R M1N", resultado_p1_verifica_p2, explicacion_p1_verifica_p2),
    ("Dr. F3R M1N verifica a Mr. _¿Sí? ¿Clarito?_", resultado_p2_verifica_p1, explicacion_p2_verifica_p1),
    ("Mensaje de Mr. _¿Sí? ¿Clarito?_ alterado", resultado_mensaje_1_alterado, explicacion_mensaje_1_alterado),
    ("Mensaje de Dr. F3R M1N alterado", resultado_mensaje_2_alterado, explicacion_mensaje_2_alterado),
    ("Llave pública incorrecta", resultado_llave_incorrecta, explicacion_llave_incorrecta),
    ("Firma mal formada", resultado_firma_mal_formada, explicacion_firma_mal_formada),
    ("Firma que no corresponde", resultado_firma_no_corresponde, explicacion_firma_no_corresponde),
]

for nombre, resultado, explicacion in pruebas:
    print(f"{nombre}: {resultado} - {explicacion}")

Mr. _¿Sí? ¿Clarito?_ verifica a Dr. F3R M1N: True - La firma es válida.
Dr. F3R M1N verifica a Mr. _¿Sí? ¿Clarito?_: True - La firma es válida.
Mensaje de Mr. _¿Sí? ¿Clarito?_ alterado: False - La firma no corresponde al mensaje o a la llave pública usada.
Mensaje de Dr. F3R M1N alterado: False - La firma no corresponde al mensaje o a la llave pública usada.
Llave pública incorrecta: False - La firma no corresponde al mensaje o a la llave pública usada.
Firma mal formada: False - La firma debe ser un número entero.
Firma que no corresponde: False - La firma no corresponde al mensaje o a la llave pública usada.


## 22. Conclusión técnica

Con esta implementación se demuestra el flujo completo de una firma digital con un esquema asimétrico.

Cada participante firma con su propia llave privada y el otro puede verificar usando la llave pública correspondiente. Si el mensaje cambia, si se usa una llave incorrecta o si la firma no pertenece al mensaje, la verificación falla.

También queda claro que esto es una implementación educativa. Sirve para entender la lógica de RSA, pero no para usarse como seguridad real, porque no incluye padding criptográfico seguro ni generación aleatoria criptográficamente segura.

## 23. Ampliación del proyecto: comunicación segura con paquete JSON

A partir de este punto se agrega lo nuevo del proyecto final sin borrar lo anterior.

La parte que ya existía sirve para generar llaves RSA, preparar mensajes, firmar y verificar. Ahora se aumenta el flujo para que exista un emisor y un receptor, donde el emisor cifra y firma un mensaje, lo empaqueta en JSON, y el receptor lo lee, lo descifra y valida si corresponde al emisor indicado.

La idea es que el mensaje no viaje visible directamente. Por eso se usa una clave de sesión educativa para cifrar el mensaje, y esa clave se protege con RSA usando la llave pública del receptor.

## 24. Nuevos módulos permitidos y carpetas para JSON

Se agregan módulos generales permitidos por la actividad. `json` se usa para guardar paquetes de transmisión, `os` para generar una clave de sesión, `time` para la fecha del paquete y `uuid` para identificar mensajes.

También se crean carpetas nuevas sin cambiar las carpetas anteriores. Las evidencias anteriores siguen igual, y ahora se agregan paquetes JSON y evidencias del flujo de comunicación segura.

In [22]:
import json
import os
import time
import uuid
import copy

CARPETA_PAQUETES_JSON = Path("paquetes_json")
CARPETA_COMUNICACION_SEGURA = CARPETA_EVIDENCIAS / "comunicacion_segura"
CARPETA_RECEPCION_JSON = CARPETA_EVIDENCIAS / "recepcion_json"
CARPETA_INTRUSO = CARPETA_EVIDENCIAS / "intruso"

for carpeta in [
    CARPETA_PAQUETES_JSON,
    CARPETA_COMUNICACION_SEGURA,
    CARPETA_RECEPCION_JSON,
    CARPETA_INTRUSO
]:
    carpeta.mkdir(parents=True, exist_ok=True)


def guardar_json(ruta, contenido):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(json.dumps(contenido, indent=4, ensure_ascii=False), encoding="utf-8")
    print(f"Archivo generado: {ruta}")


def cargar_json(ruta):
    return json.loads(Path(ruta).read_text(encoding="utf-8"))

## 25. Cifrado educativo del mensaje con clave de sesión

El mensaje se cifra usando una clave de sesión educativa. Esto se hace para que el contenido no aparezca en texto claro dentro del archivo JSON.

El mecanismo usado es XOR con un flujo de bytes derivado de SHA-256. No es un cifrado industrial como AES, pero cumple el objetivo educativo: transformar el mensaje en algo no legible y recuperarlo después usando la misma clave de sesión.

La clave de sesión no se manda en claro. Después se cifra con RSA usando la llave pública del receptor.

In [23]:
def generar_clave_sesion(tam_bytes=32):
    return os.urandom(tam_bytes)


def generar_flujo_clave(clave_sesion, longitud):
    flujo = b""
    contador = 0

    while len(flujo) < longitud:
        bloque = hashlib.sha256(clave_sesion + contador.to_bytes(4, "big")).digest()
        flujo += bloque
        contador += 1

    return flujo[:longitud]


def aplicar_xor(datos, flujo):
    return bytes(byte ^ flujo_byte for byte, flujo_byte in zip(datos, flujo))


def cifrar_mensaje_con_clave_sesion(mensaje, clave_sesion):
    if not isinstance(mensaje, str):
        raise TypeError("El mensaje debe ser texto.")

    mensaje_bytes = mensaje.encode("utf-8")
    flujo = generar_flujo_clave(clave_sesion, len(mensaje_bytes))
    mensaje_cifrado = aplicar_xor(mensaje_bytes, flujo)

    return mensaje_cifrado.hex()


def descifrar_mensaje_con_clave_sesion(mensaje_cifrado_hex, clave_sesion):
    try:
        mensaje_cifrado = bytes.fromhex(mensaje_cifrado_hex)
    except ValueError:
        raise ValueError("El mensaje cifrado no tiene formato hexadecimal válido.")

    flujo = generar_flujo_clave(clave_sesion, len(mensaje_cifrado))
    mensaje_bytes = aplicar_xor(mensaje_cifrado, flujo)

    try:
        return mensaje_bytes.decode("utf-8")
    except UnicodeDecodeError:
        raise ValueError("El mensaje descifrado no pudo interpretarse como texto UTF-8.")

## 26. Protección de la clave de sesión usando RSA

La clave de sesión sirve para cifrar y descifrar el mensaje, pero no debe viajar visible en el paquete JSON.

Para protegerla se usa RSA. El emisor cifra la clave de sesión con la llave pública del receptor. Después el receptor usa su llave privada para recuperarla.

Así se combinan dos ideas: la clave de sesión cifra el mensaje, y RSA protege esa clave para que solo el receptor correcto pueda recuperarla.

In [24]:
def cifrar_clave_sesion_rsa(clave_sesion, llave_publica_receptor):
    clave_entero = int.from_bytes(clave_sesion, "big")
    e = llave_publica_receptor["e"]
    n = llave_publica_receptor["n"]

    if clave_entero >= n:
        raise ValueError("La clave de sesión es demasiado grande para la llave RSA del receptor.")

    return pow(clave_entero, e, n)


def descifrar_clave_sesion_rsa(clave_sesion_cifrada, llave_privada_receptor, tam_bytes=32):
    d = llave_privada_receptor["d"]
    n = llave_privada_receptor["n"]
    clave_entero = pow(clave_sesion_cifrada, d, n)

    try:
        return clave_entero.to_bytes(tam_bytes, "big")
    except OverflowError:
        raise ValueError("La clave recuperada no tiene el tamaño esperado.")

## 27. Conversión de llaves públicas para el paquete JSON

El paquete JSON debe incluir la llave pública del emisor para que el receptor pueda verificar la firma.

La llave privada nunca se guarda dentro del JSON. Si apareciera un valor privado como `d`, el paquete se rechaza.

In [25]:
def convertir_llave_publica_a_json(llave_publica):
    return {
        "e": str(llave_publica["e"]),
        "n": str(llave_publica["n"])
    }


def convertir_llave_publica_desde_json(llave_publica_json):
    if not isinstance(llave_publica_json, dict):
        raise ValueError("La llave pública del emisor debe ser un objeto JSON.")

    if "e" not in llave_publica_json or "n" not in llave_publica_json:
        raise ValueError("La llave pública debe contener e y n.")

    if "d" in llave_publica_json:
        raise ValueError("El paquete JSON no debe incluir valores de llave privada.")

    return {
        "e": int(llave_publica_json["e"]),
        "n": int(llave_publica_json["n"])
    }

## 28. Creación del paquete JSON de transmisión

El emisor crea un paquete JSON con el mensaje cifrado, la clave de sesión cifrada, la firma y su llave pública.

El paquete representa lo que viajaría del emisor al receptor. No contiene el mensaje en texto claro ni llaves privadas.

In [26]:
def crear_paquete_transmision(sender, receiver, mensaje, llave_privada_emisor, llave_publica_emisor, llave_publica_receptor):
    firma, preparado = firmar_mensaje(mensaje, llave_privada_emisor)
    clave_sesion = generar_clave_sesion()
    mensaje_cifrado = cifrar_mensaje_con_clave_sesion(mensaje, clave_sesion)
    clave_sesion_cifrada = cifrar_clave_sesion_rsa(clave_sesion, llave_publica_receptor)

    paquete = {
        "package_version": "1.0-educativo",
        "message_id": str(uuid.uuid4()),
        "created_at": int(time.time()),
        "sender": sender,
        "receiver": receiver,
        "encrypted_message": mensaje_cifrado,
        "encrypted_session_key": str(clave_sesion_cifrada),
        "signature": str(firma),
        "hash_algorithm": "SHA-256",
        "encryption_type": "XOR educativo con flujo derivado de SHA-256",
        "session_key_protection": "RSA con llave pública del receptor",
        "sender_public_key": convertir_llave_publica_a_json(llave_publica_emisor)
    }

    evidencia = {
        "mensaje_original": mensaje,
        "hash_mensaje": preparado["hash_hexadecimal"],
        "valor_preparado": preparado["valor_preparado"],
        "firma": firma,
        "clave_sesion_hex": clave_sesion.hex(),
        "mensaje_cifrado": mensaje_cifrado,
        "clave_sesion_cifrada": clave_sesion_cifrada
    }

    return paquete, evidencia

## 29. Recepción, descifrado y verificación del paquete JSON

El receptor lee el paquete JSON, revisa que tenga los campos necesarios, recupera la clave de sesión con su llave privada, descifra el mensaje y verifica la firma usando la llave pública del emisor.

El resultado no se deja solo como verdadero o falso. También se guarda una explicación para poder defender qué ocurrió.

In [27]:
def validar_estructura_paquete(paquete):
    if not isinstance(paquete, dict):
        return False, "El paquete JSON debe ser un objeto."

    campos_obligatorios = [
        "sender",
        "receiver",
        "encrypted_message",
        "encrypted_session_key",
        "signature",
        "hash_algorithm",
        "sender_public_key"
    ]

    faltantes = [campo for campo in campos_obligatorios if campo not in paquete]

    if faltantes:
        return False, f"El paquete JSON está incompleto. Faltan campos: {', '.join(faltantes)}."

    if paquete.get("hash_algorithm") != "SHA-256":
        return False, "El algoritmo hash indicado no es compatible. Se esperaba SHA-256."

    if not isinstance(paquete.get("sender_public_key"), dict):
        return False, "La llave pública del emisor debe venir como objeto JSON."

    if "d" in paquete.get("sender_public_key", {}):
        return False, "El paquete JSON no debe contener valores de llave privada."

    return True, "La estructura básica del paquete JSON es válida."


def recibir_y_validar_paquete(paquete, llave_privada_receptor):
    estructura_valida, explicacion_estructura = validar_estructura_paquete(paquete)

    resultado = {
        "estructura_valida": estructura_valida,
        "mensaje_valido": False,
        "mensaje_descifrado": None,
        "sender": paquete.get("sender") if isinstance(paquete, dict) else None,
        "receiver": paquete.get("receiver") if isinstance(paquete, dict) else None,
        "explicacion": explicacion_estructura,
        "detalles": {}
    }

    if not estructura_valida:
        return resultado

    try:
        clave_sesion_cifrada = int(paquete["encrypted_session_key"])
    except Exception:
        resultado["explicacion"] = "La clave de sesión cifrada no tiene formato numérico válido."
        return resultado

    try:
        firma = int(paquete["signature"])
    except Exception:
        resultado["explicacion"] = "La firma no tiene formato numérico válido."
        return resultado

    try:
        llave_publica_emisor = convertir_llave_publica_desde_json(paquete["sender_public_key"])
    except Exception as error:
        resultado["explicacion"] = f"La llave pública del emisor no pudo interpretarse: {error}"
        return resultado

    try:
        clave_sesion = descifrar_clave_sesion_rsa(clave_sesion_cifrada, llave_privada_receptor)
        resultado["detalles"]["clave_sesion_recuperada_hex"] = clave_sesion.hex()
    except Exception as error:
        resultado["explicacion"] = f"No se pudo recuperar la clave de sesión. {error}"
        return resultado

    try:
        mensaje_descifrado = descifrar_mensaje_con_clave_sesion(paquete["encrypted_message"], clave_sesion)
        resultado["mensaje_descifrado"] = mensaje_descifrado
    except Exception as error:
        resultado["explicacion"] = f"No se pudo descifrar el mensaje. {error}"
        return resultado

    firma_valida, explicacion_firma, datos_firma = verificar_firma(mensaje_descifrado, firma, llave_publica_emisor)

    resultado["mensaje_valido"] = firma_valida

    if firma_valida:
        resultado["explicacion"] = "La firma es válida. El mensaje fue descifrado, no fue alterado y corresponde al emisor indicado."
    else:
        resultado["explicacion"] = explicacion_firma

    resultado["detalles"]["verificacion_firma"] = datos_firma

    return resultado

## 30. Formatos nuevos para evidencias de comunicación segura

Estos formatos guardan en archivos de texto lo que ocurrió en el paquete JSON y en la recepción.

Sirven para mostrar el mensaje original, el mensaje cifrado, la clave de sesión cifrada, el resultado de descifrado y la verificación de la firma.

In [28]:
def formato_paquete_seguro(titulo, paquete, evidencia):
    return f'''{titulo}

Mensaje original:
{evidencia["mensaje_original"]}

Hash SHA-256 del mensaje:
{evidencia["hash_mensaje"]}

Valor preparado para firma:
{evidencia["valor_preparado"]}

Firma generada con la llave privada del emisor:
{evidencia["firma"]}

Clave de sesión en hexadecimal, solo evidencia local:
{evidencia["clave_sesion_hex"]}

Mensaje cifrado guardado en JSON:
{evidencia["mensaje_cifrado"]}

Clave de sesión cifrada con RSA:
{evidencia["clave_sesion_cifrada"]}

Emisor:
{paquete["sender"]}

Receptor:
{paquete["receiver"]}

Tipo de cifrado:
{paquete["encryption_type"]}

Protección de clave:
{paquete["session_key_protection"]}

Nota:
El paquete JSON no contiene llaves privadas.
'''


def formato_recepcion(titulo, resultado):
    detalles = json.dumps(resultado.get("detalles", {}), indent=4, ensure_ascii=False)

    return f'''{titulo}

Emisor indicado:
{resultado.get("sender")}

Receptor indicado:
{resultado.get("receiver")}

Estructura válida:
{resultado.get("estructura_valida")}

Mensaje válido:
{resultado.get("mensaje_valido")}

Mensaje descifrado:
{resultado.get("mensaje_descifrado")}

Explicación:
{resultado.get("explicacion")}

Detalles técnicos:
{detalles}
'''

## 31. Generación de usuario intruso para pruebas inválidas

Se agrega un tercer usuario solo para probar errores.

El intruso no reemplaza a los usuarios anteriores. Solo sirve para demostrar qué ocurre si se usa una llave pública incorrecta o una llave privada que no corresponde al receptor real.

In [29]:
llave_publica_3, llave_privada_3, detalles_3 = generar_llaves(bits=256)

guardar_txt(CARPETA_INTRUSO / "01_llave_publica_intruso.txt", formato_llave_publica("Intruso", llave_publica_3))
guardar_txt(CARPETA_INTRUSO / "02_llave_privada_intruso.txt", formato_llave_privada("Intruso", llave_privada_3))
guardar_txt(CARPETA_INTRUSO / "03_detalles_generacion_intruso.txt", formato_detalles("Intruso", detalles_3))

print("Llaves generadas correctamente para el usuario intruso.")

Archivo generado: evidencias/intruso/01_llave_publica_intruso.txt
Archivo generado: evidencias/intruso/02_llave_privada_intruso.txt
Archivo generado: evidencias/intruso/03_detalles_generacion_intruso.txt
Llaves generadas correctamente para el usuario intruso.


## 32. Paquete JSON válido y paquete con mensaje vacío

Aquí se generan los dos primeros paquetes.

El paquete válido representa el caso ideal. El paquete con mensaje vacío demuestra que incluso una cadena vacía puede cifrarse, firmarse, enviarse, descifrarse y verificarse.

In [30]:
emisor = "Mr. _¿Sí? ¿Clarito?_"
receptor = "Dr. F3R M1N"

mensaje_seguro = "Estimado Dr. F3R M1N, este mensaje viaja cifrado, firmado y dentro de un paquete JSON."
mensaje_seguro_vacio = ""

paquete_valido, evidencia_paquete_valido = crear_paquete_transmision(
    emisor,
    receptor,
    mensaje_seguro,
    llave_privada_1,
    llave_publica_1,
    llave_publica_2
)

paquete_mensaje_vacio, evidencia_paquete_vacio = crear_paquete_transmision(
    emisor,
    receptor,
    mensaje_seguro_vacio,
    llave_privada_1,
    llave_publica_1,
    llave_publica_2
)

guardar_json(CARPETA_PAQUETES_JSON / "01_paquete_valido.json", paquete_valido)
guardar_json(CARPETA_PAQUETES_JSON / "02_paquete_mensaje_vacio.json", paquete_mensaje_vacio)

guardar_txt(CARPETA_COMUNICACION_SEGURA / "01_evidencia_paquete_valido.txt", formato_paquete_seguro("Paquete JSON válido", paquete_valido, evidencia_paquete_valido))
guardar_txt(CARPETA_COMUNICACION_SEGURA / "02_evidencia_paquete_mensaje_vacio.txt", formato_paquete_seguro("Paquete JSON con mensaje vacío", paquete_mensaje_vacio, evidencia_paquete_vacio))

print("Paquete válido y paquete con mensaje vacío generados correctamente.")

Archivo generado: paquetes_json/01_paquete_valido.json
Archivo generado: paquetes_json/02_paquete_mensaje_vacio.json
Archivo generado: evidencias/comunicacion_segura/01_evidencia_paquete_valido.txt
Archivo generado: evidencias/comunicacion_segura/02_evidencia_paquete_mensaje_vacio.txt
Paquete válido y paquete con mensaje vacío generados correctamente.


## 33. Paquetes JSON alterados o inválidos

En esta sección se crean paquetes para probar los errores pedidos.

Se genera un mensaje alterado después de firmar, una firma alterada, una llave pública incorrecta, un caso para llave privada incorrecta, un JSON incompleto, un JSON con datos mal formados y una firma que no corresponde al mensaje recibido.

In [31]:
clave_sesion_valida = bytes.fromhex(evidencia_paquete_valido["clave_sesion_hex"])

paquete_mensaje_alterado = copy.deepcopy(paquete_valido)
mensaje_seguro_alterado = "Este mensaje fue alterado después de haber generado la firma."
paquete_mensaje_alterado["encrypted_message"] = cifrar_mensaje_con_clave_sesion(mensaje_seguro_alterado, clave_sesion_valida)
guardar_json(CARPETA_PAQUETES_JSON / "03_paquete_mensaje_alterado.json", paquete_mensaje_alterado)

paquete_firma_alterada = copy.deepcopy(paquete_valido)
paquete_firma_alterada["signature"] = str(int(paquete_firma_alterada["signature"]) + 12345)
guardar_json(CARPETA_PAQUETES_JSON / "04_paquete_firma_alterada.json", paquete_firma_alterada)

paquete_llave_publica_incorrecta = copy.deepcopy(paquete_valido)
paquete_llave_publica_incorrecta["sender_public_key"] = convertir_llave_publica_a_json(llave_publica_3)
guardar_json(CARPETA_PAQUETES_JSON / "05_paquete_llave_publica_incorrecta.json", paquete_llave_publica_incorrecta)

paquete_para_llave_privada_incorrecta = copy.deepcopy(paquete_valido)
guardar_json(CARPETA_PAQUETES_JSON / "06_paquete_para_llave_privada_incorrecta.json", paquete_para_llave_privada_incorrecta)

paquete_incompleto = copy.deepcopy(paquete_valido)
paquete_incompleto.pop("signature", None)
guardar_json(CARPETA_PAQUETES_JSON / "07_paquete_incompleto.json", paquete_incompleto)

paquete_datos_mal_formados = copy.deepcopy(paquete_valido)
paquete_datos_mal_formados["encrypted_session_key"] = "no_es_un_numero"
paquete_datos_mal_formados["encrypted_message"] = "texto-no-hexadecimal"
guardar_json(CARPETA_PAQUETES_JSON / "08_paquete_datos_mal_formados.json", paquete_datos_mal_formados)

firma_otro_mensaje, preparado_otro_mensaje = firmar_mensaje("Mensaje diferente para probar firma que no corresponde.", llave_privada_1)
paquete_firma_no_corresponde = copy.deepcopy(paquete_valido)
paquete_firma_no_corresponde["signature"] = str(firma_otro_mensaje)
guardar_json(CARPETA_PAQUETES_JSON / "09_paquete_firma_no_corresponde.json", paquete_firma_no_corresponde)

print("Paquetes inválidos generados correctamente.")

Archivo generado: paquetes_json/03_paquete_mensaje_alterado.json
Archivo generado: paquetes_json/04_paquete_firma_alterada.json
Archivo generado: paquetes_json/05_paquete_llave_publica_incorrecta.json
Archivo generado: paquetes_json/06_paquete_para_llave_privada_incorrecta.json
Archivo generado: paquetes_json/07_paquete_incompleto.json
Archivo generado: paquetes_json/08_paquete_datos_mal_formados.json
Archivo generado: paquetes_json/09_paquete_firma_no_corresponde.json
Paquetes inválidos generados correctamente.


## 34. Recepción y validación de todos los paquetes JSON

Ahora el receptor intenta procesar cada paquete.

En la mayoría de los casos inválidos el mensaje no debe aceptarse. Esto demuestra que el sistema no solo funciona con el caso ideal, también detecta alteraciones, errores de estructura y llaves incorrectas.

In [32]:
pruebas_json = [
    ("01_resultado_paquete_valido.txt", "Resultado del paquete válido", paquete_valido, llave_privada_2),
    ("02_resultado_mensaje_vacio.txt", "Resultado del paquete con mensaje vacío", paquete_mensaje_vacio, llave_privada_2),
    ("03_resultado_mensaje_alterado.txt", "Resultado del paquete con mensaje alterado después de firmar", paquete_mensaje_alterado, llave_privada_2),
    ("04_resultado_firma_alterada.txt", "Resultado del paquete con firma alterada", paquete_firma_alterada, llave_privada_2),
    ("05_resultado_llave_publica_incorrecta.txt", "Resultado del paquete con llave pública incorrecta", paquete_llave_publica_incorrecta, llave_privada_2),
    ("06_resultado_llave_privada_incorrecta.txt", "Resultado al intentar descifrar con llave privada incorrecta", paquete_para_llave_privada_incorrecta, llave_privada_3),
    ("07_resultado_paquete_incompleto.txt", "Resultado del paquete JSON incompleto", paquete_incompleto, llave_privada_2),
    ("08_resultado_datos_mal_formados.txt", "Resultado del paquete JSON con datos mal formados", paquete_datos_mal_formados, llave_privada_2),
    ("09_resultado_firma_no_corresponde.txt", "Resultado del paquete con firma que no corresponde", paquete_firma_no_corresponde, llave_privada_2)
]

resumen_json = []

for archivo, titulo, paquete, llave_privada_usada in pruebas_json:
    resultado = recibir_y_validar_paquete(paquete, llave_privada_usada)
    guardar_txt(CARPETA_RECEPCION_JSON / archivo, formato_recepcion(titulo, resultado))
    resumen_json.append({
        "prueba": titulo,
        "estructura_valida": resultado["estructura_valida"],
        "mensaje_valido": resultado["mensaje_valido"],
        "mensaje_descifrado": resultado["mensaje_descifrado"],
        "explicacion": resultado["explicacion"]
    })

guardar_json(CARPETA_RECEPCION_JSON / "10_resumen_resultados_json.json", resumen_json)

for prueba in resumen_json:
    print(f'{prueba["prueba"]}: estructura={prueba["estructura_valida"]}, valido={prueba["mensaje_valido"]}')

Archivo generado: evidencias/recepcion_json/01_resultado_paquete_valido.txt
Archivo generado: evidencias/recepcion_json/02_resultado_mensaje_vacio.txt
Archivo generado: evidencias/recepcion_json/03_resultado_mensaje_alterado.txt
Archivo generado: evidencias/recepcion_json/04_resultado_firma_alterada.txt
Archivo generado: evidencias/recepcion_json/05_resultado_llave_publica_incorrecta.txt
Archivo generado: evidencias/recepcion_json/06_resultado_llave_privada_incorrecta.txt
Archivo generado: evidencias/recepcion_json/07_resultado_paquete_incompleto.txt
Archivo generado: evidencias/recepcion_json/08_resultado_datos_mal_formados.txt
Archivo generado: evidencias/recepcion_json/09_resultado_firma_no_corresponde.txt
Archivo generado: evidencias/recepcion_json/10_resumen_resultados_json.json
Resultado del paquete válido: estructura=True, valido=True
Resultado del paquete con mensaje vacío: estructura=True, valido=True
Resultado del paquete con mensaje alterado después de firmar: estructura=Tru

## 35. Resumen final del proyecto ampliado

El proyecto ahora conserva la firma y verificación RSA anterior, pero también agrega comunicación segura con paquete JSON.

El emisor cifra el mensaje con una clave de sesión, cifra esa clave con RSA usando la llave pública del receptor y firma el mensaje con su llave privada. El receptor usa su llave privada para recuperar la clave de sesión, descifra el mensaje y verifica la firma con la llave pública del emisor.

La implementación sigue siendo educativa. Sirve para explicar cómo se conectan cifrado, firma, JSON, descifrado y validación, pero no debe usarse como seguridad real porque no implementa un protocolo industrial.